# HS524M — Urban Informatics and Spatial Analytics

## Lectures 15–17: Spatial Joins and Ward-Level Mapping with GeoPandas

### Case study: Points of Interest and Municipal Wards in Delhi

In the earlier lectures, we learned how to:

- work with tabular data using Pandas;
- create and inspect GeoDataFrames;
- understand points and polygons;
- work with coordinate reference systems (CRS);
- reproject spatial data;
- plot spatial layers;
- add basic cartographic elements.

In these lectures, we move from **mapping where things are** to asking:

> **Which spatial unit does each feature belong to?**

We will use two datasets:

- `delhi_pois.geojson` — Points of Interest in Delhi
- `delhi_wards.geojson` — Municipal ward polygons in Delhi

The main idea across Lectures 15–17 is the **spatial join**.

We will move slowly from:

```text
normal Pandas merge
        ↓
spatial join
        ↓
count POIs by ward
        ↓
merge results back to ward polygons
        ↓
create ward-level choropleth maps
        ↓
compare counts with density
```

## File path used in this notebook

The path below is the location of the data **on the instructor's computer**:

```python
data_path = r"C:\Teaching\IIT Tirupati\UISA\2026\uisa_codes\data"
```

Students must replace this with the location where the data are stored **on their own system**.

For example, if the files are stored in:

```text
D:\UISA\data
```

then use:

```python
data_path = r"D:\UISA\data"
```

The important point is:

> The folder path depends on where the files are stored on your own computer.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

data_path = r"C:\Teaching\IIT Tirupati\UISA\2026\uisa_codes\data"

pois = gpd.read_file(data_path + r"\delhi_pois.geojson")
wards = gpd.read_file(data_path + r"\delhi_wards.geojson")

## First inspect the two datasets

Before performing any spatial operation, inspect:

- the first few rows;
- the column names;
- number of observations;
- geometry type;
- CRS.

A GeoDataFrame contains both:

1. **attributes** — ordinary columns such as name, type, category;
2. **geometry** — the spatial information.

In [ ]:
print("POIs:")
display(pois.head())

print("Wards:")
display(wards.head())

print("Number of POIs:", len(pois))
print("Number of wards:", len(wards))

print("POI CRS:", pois.crs)
print("Ward CRS:", wards.crs)

print("POI geometry type:")
print(pois.geometry.geom_type.value_counts())

print("Ward geometry type:")
print(wards.geometry.geom_type.value_counts())

## Prepare simpler column names

The original files contain column names inherited from their source datasets.

For these lectures, we will use shorter names where useful.

The POI data contain:

- `sl_no` — identifier;
- `PoI_Type` — type of POI;
- geometry.

The ward data contain fields identifying each municipal ward.

Run the following cells and inspect the resulting columns before continuing.

In [ ]:
print(pois.columns)
print(wards.columns)

In [ ]:
pois = pois.rename(columns={
    "sl_no": "id",
    "PoI_Type": "poi_type"
})

wards = wards.rename(columns={
    "OBJECTID": "ward_id",
    "MWARD_NM": "ward_name"
})

display(pois.head())
display(wards.head())

## Create a broader POI category

The original `poi_type` field contains many detailed POI types.

For some analyses, it is useful to group these into broader categories such as:

- Education
- Health
- Finance
- Green space
- Commerce
- Transport & Infrastructure
- Public Services & Utilities
- Leisure, Tourism & Culture

We can do this using a Python dictionary and `.map()`, which links each original POI type to a broader category.

In [ ]:
mapping = {
    "Schools": "Education",
    "ITI_NIC_2018": "Education",
    "Library_Open_Source": "Education",

    "Hospitals": "Health",
    "Pharmacetucal Labs": "Health",
    "Pharmacy": "Health",
    "Psychological_Counselling_Open_Source": "Health",
    "Veterinary Hospital": "Health",

    "ATM": "Finance",
    "Bank Branch": "Finance",
    "Bank Mitra": "Finance",
    "CSC_NIC_Through_API": "Finance",

    "Parks": "Green space",
    "Playgrounds_Open_Source": "Green space",
    "National_Parks_and_Bird_Sanctuaries_Open_Source": "Green space",

    "shop": "Commerce",
    "Shopping_Malls_Open_Source": "Commerce",
    "Weekly_Markets_Open_Source": "Commerce",

    "DMRC Stations": "Transport & Infrastructure",
    "Railway Junctions": "Transport & Infrastructure",
    "Level_Crossing_Open_Source": "Transport & Infrastructure",
    "Airports Helipads": "Transport & Infrastructure",
    "Aerodrome": "Transport & Infrastructure",
    "Transport_Nagar_Open_Source": "Transport & Infrastructure",
    "Parking_Open_Source": "Transport & Infrastructure",

    "Post Offices": "Public Services & Utilities",
    "office": "Public Services & Utilities",
    "Police_Station_Open_Source": "Public Services & Utilities",
    "Fire_Stations_Open_Source": "Public Services & Utilities",
    "Fuel_Stations_Open_Source": "Public Services & Utilities",
    "Toilets_Open_Source": "Public Services & Utilities",

    "Sports Facilities": "Leisure, Tourism & Culture",
    "Stadium": "Leisure, Tourism & Culture",
    "Golf_Course_Open_Source": "Leisure, Tourism & Culture",
    "Cinemas_Open_Source": "Leisure, Tourism & Culture",
    "tourism": "Leisure, Tourism & Culture",
    "Heritage & Tourism": "Leisure, Tourism & Culture",
    "historic": "Leisure, Tourism & Culture",
    "Museum_Tourism_Department": "Leisure, Tourism & Culture",
    "Lodging_Open_Source": "Leisure, Tourism & Culture"
}

pois["category"] = pois["poi_type"].map(mapping).fillna("Other")

pois["category"].value_counts()

# Lecture 15 — From a Normal Pandas Join to a Spatial Join

Before learning a spatial join, it helps to revisit something familiar: a normal Pandas merge.

A normal merge connects two tables using a **common attribute**.

Suppose we have one table containing ward names:

In [ ]:
ward = pd.DataFrame({
    "ward_id": [1, 2, 3],
    "ward_name": ["Ward A", "Ward B", "Ward C"]
})

ward

And another table containing population:

In [ ]:
population = pd.DataFrame({
    "ward_id": [1, 2, 3],
    "population": [42000, 57000, 38000]
})

population

Both tables contain a common column:

```text
ward_id
```

That common column acts as the **key**.

Pandas can therefore match:

```text
ward_id = 1
```

in the first table with:

```text
ward_id = 1
```

in the second table.

In [ ]:
ward_data = ward.merge(
    population,
    on="ward_id"
)

ward_data

The result is:

```text
ward_id   ward_name   population
1         Ward A      42000
2         Ward B      57000
3         Ward C      38000
```

The key idea is:

> **A normal Pandas merge joins rows using matching attribute values.**

Conceptually:

```text
WARD TABLE                 POPULATION TABLE

ward_id  ward_name         ward_id  population
   1      Ward A              1       42000
   2      Ward B              2       57000
   3      Ward C              3       38000

              ↓ match using ward_id ↓

ward_id  ward_name  population
   1      Ward A       42000
   2      Ward B       57000
   3      Ward C       38000
```

## What does `how="left"` mean?

Consider a slightly different population table where Ward C is missing.

In [ ]:
population2 = pd.DataFrame({
    "ward_id": [1, 2],
    "population": [42000, 57000]
})

ward.merge(
    population2,
    on="ward_id",
    how="left"
)

With:

```python
how="left"
```

we are saying:

> **Keep all rows from the left DataFrame.**

Therefore Ward C remains in the output even though there is no matching population record.

Its population becomes missing (`NaN`).

This idea becomes important when we use a spatial join.

# Why do we need a spatial join?

Now look at our Delhi POI data.

A POI may contain:

```text
id
poi_type
category
geometry
```

The ward data may contain:

```text
ward_id
ward_name
geometry
```

But the POI data do not necessarily contain a `ward_id`.

Therefore we cannot simply write:

```python
pois.merge(wards, on="ward_id")
```

because the common key does not yet exist.

However, the POI **does have a location**.

So we can ask:

> **Which ward polygon contains this POI point?**

This is where a **spatial join** becomes useful.

Instead of matching:

```text
attribute = attribute
```

we match using:

```text
spatial location → spatial relationship → polygon
```

## Normal join versus spatial join

| Normal Pandas merge | GeoPandas spatial join |
|---|---|
| Uses attributes | Uses geometry |
| Needs a common key | Needs a spatial relationship |
| Example: same `ward_id` | Example: point lies within ward |
| `.merge()` | `gpd.sjoin()` |

In simple terms:

> **Pandas asks: Do these rows have the same ID?**

> **GeoPandas asks: Do these features have the required spatial relationship?**

## Spatial relationships

For a point and a polygon, common spatial relationships include:

### `within`

Ask:

> Is this point inside this polygon?

```text
+----------------------+
|                      |
|          •           |
|                      |
+----------------------+
```

The point is **within** the polygon.

### `contains`

Look at the same relationship from the polygon's perspective:

> Does this polygon contain this point?

### `intersects`

Ask:

> Do the two geometries share any location?

For our current problem, we want:

> **Which ward is each POI within?**

Therefore we use:

```python
predicate="within"
```

## CRS must match before a spatial join

Before comparing the location of two spatial layers, they should use the same CRS.

Check:

In [ ]:
print(pois.crs)
print(wards.crs)

If both are already in the same CRS, we can proceed.

If they are different, one layer must be reprojected so that both use the same coordinate system.

For the spatial join itself, EPSG:4326 is acceptable because we are testing a topological relationship — whether a point lies inside a polygon.

We do **not** need metre-based coordinates simply to perform a point-in-polygon join.

## Perform the spatial join

We want:

- POIs on the **left**;
- wards on the **right**;
- all POIs retained;
- the ward information attached where the POI lies inside a ward.

In [ ]:
joined = gpd.sjoin(
    pois,
    wards[["ward_id", "ward_name", "geometry"]],
    how="left",
    predicate="within"
)

joined.head()

## Understanding the code line by line

```python
gpd.sjoin(
```

asks GeoPandas to perform a spatial join.

```python
pois,
```

is the **left GeoDataFrame**.

These are the rows we want to keep.

```python
wards[["ward_id", "ward_name", "geometry"]],
```

is the **right GeoDataFrame**.

We only pass the ward fields that we actually need.

```python
how="left",
```

means:

> Keep every POI, even if it does not find a matching ward.

```python
predicate="within"
```

means:

> Match a POI to a ward when the POI point lies inside the ward polygon.

## What changed after the join?

Before the spatial join, a POI did not know its ward.

After the join, the ward attributes have been attached to the point.

The point geometry itself remains the same.

Let us compare the columns.

In [ ]:
print("Before join:")
print(pois.columns)

print("\nAfter join:")
print(joined.columns)

GeoPandas also adds a column called:

```text
index_right
```

This stores the index of the matching row from the right-hand GeoDataFrame.

For our current analysis we do not need it, so we can remove it.

In [ ]:
joined = joined.drop(columns="index_right")

joined.head()

## Check unmatched POIs

Because we used a left join, POIs that did not fall inside any ward remain in the output.

For those rows, `ward_name` will be missing.

In [ ]:
joined["ward_name"].isna().sum()

Do not automatically delete unmatched points.

They may occur because:

- the point lies outside the ward boundary;
- the ward dataset and POI dataset have different spatial coverage;
- the point lies very close to a boundary;
- there may be a problem in the source data.

The important lesson is:

> **Spatial analysis should include checking the result, not just running the command.**

## Class Exercise 12

Using only POIs where:

```python
category == "Education"
```

1. Create a new GeoDataFrame containing only Education POIs.
2. Spatially join these points to the Delhi wards.
3. Display the columns `poi_type`, `ward_id`, and `ward_name`.
4. Count how many Education POIs did not match a ward.

**Question:** Why was a spatial join necessary here rather than a normal Pandas merge?

# Lecture 16 — From Spatial Join to Ward-Level Information

In Lecture 15, we answered:

> **Which ward does each POI belong to?**

Now every POI has a ward name attached to it.

That allows us to ask a different kind of question:

> **How many POIs are located in each ward?**

Notice that the spatial operation has already been completed.

The next steps use familiar Pandas operations.

## Start with `value_counts()`

The simplest way to count how many POIs occur in each ward is:

In [ ]:
joined["ward_name"].value_counts().head(10)

This tells us which ward names appear most frequently in the spatially joined dataset.

Since one row represents one POI:

> **Number of rows for a ward = number of matched POIs in that ward.**

This is a good example of how a spatial operation can create the information needed for ordinary tabular analysis.

## Count using `groupby()`

We can also create a proper DataFrame containing one row per ward.

In [ ]:
counts = (
    joined
    .dropna(subset=["ward_name"])
    .groupby(["ward_id", "ward_name"])
    .size()
    .reset_index(name="num_pois")
)

counts.head()

Let us unpack this.

```python
.dropna(subset=["ward_name"])
```

removes POIs that did not match a ward.

```python
.groupby(["ward_id", "ward_name"])
```

groups all POIs belonging to the same ward.

```python
.size()
```

counts how many rows are in each group.

```python
.reset_index(name="num_pois")
```

turns the grouped result back into a normal DataFrame and names the new count column:

```text
num_pois
```

In [ ]:
counts.sort_values("num_pois", ascending=False).head(10)

## The count table is no longer a map

At this stage, `counts` is just a normal table:

```text
ward_id   ward_name   num_pois
...
```

It contains no geometry.

So how can we map these values?

We already have the ward polygons in:

```python
wards
```

and both datasets contain:

```text
ward_id
```

This means we can now return to a **normal attribute merge**.

In [ ]:
ward_map = wards.merge(
    counts[["ward_id", "num_pois"]],
    on="ward_id",
    how="left"
)

ward_map.head()

This is an important full-circle workflow:

```text
POIs + Wards
     ↓
SPATIAL JOIN
using location
     ↓
Each POI receives a ward ID
     ↓
GROUP / COUNT
     ↓
Count table by ward
     ↓
NORMAL MERGE
using ward_id
     ↓
Ward polygons receive the count
```

So both kinds of joins are useful:

- spatial join when location is the relationship;
- normal merge when a shared attribute is available.

## Why use `how="left"` when merging back to wards?

Suppose one ward contains no matched POIs.

If we use:

```python
how="left"
```

with `wards` on the left, that ward is still retained.

Its `num_pois` value will initially be missing.

But logically, if the ward is in the dataset and no POIs were counted inside it, the count should be zero.

In [ ]:
ward_map["num_pois"] = ward_map["num_pois"].fillna(0)

ward_map[["ward_name", "num_pois"]].head()

## First ward-level thematic map

Until now, we have mostly plotted:

- point locations;
- polygon boundaries;
- categories of points.

Now each ward polygon has a numerical value:

```text
num_pois
```

We can colour each polygon according to this value.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

ward_map.plot(
    ax=ax,
    column="num_pois",
    legend=True,
    edgecolor="white"
)

ax.set_title("Number of Mapped POIs by Ward")
ax.set_axis_off()

plt.show()

## What does `column="num_pois"` mean?

The argument:

```python
column="num_pois"
```

tells GeoPandas:

> Colour each ward according to the value stored in its `num_pois` column.

A ward with one value receives one colour, while a ward with a different value receives another.

Because `num_pois` is numerical, the map uses a continuous colour scale rather than separate category colours.

This kind of map is commonly called a **choropleth map**.

## Class Exercise 13

Using the `joined` GeoDataFrame:

1. Select only POIs where `category == "Health"`.
2. Count the number of Health POIs in each ward.
3. Display the five wards with the largest number of Health POIs.
4. Merge the Health counts back to the ward polygons.
5. Replace missing counts with `0`.

Do not create the final map yet.

The focus of this exercise is the sequence:

```text
filter → count → merge
```

# Lecture 17 — Choropleth Mapping: Counts and Density

In Lecture 16, we created a ward-level variable:

```text
num_pois
```

Now we will focus on how to map and interpret such values correctly.

A choropleth map uses colour to represent a value associated with each geographic area.

Examples include:

- population by district;
- crime rate by ward;
- unemployment rate by state;
- average income by census tract;
- number of mapped POIs by ward.

The map does not show the original point locations.

Instead, it shows a **summary value attached to each polygon**.

## A cleaner choropleth map

We can control the colour palette using:

```python
cmap=
```

For example:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

ward_map.plot(
    ax=ax,
    column="num_pois",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5
)

ax.set_title("Number of Mapped POIs by Ward")
ax.set_axis_off()

plt.show()

## Interpreting the colour scale

For a numerical variable such as `num_pois`, the colours represent a progression from lower to higher values.

The legend tells us how the colours correspond to the underlying numbers.

The colour itself does not have meaning unless we connect it to the variable.

For example:

> A dark-coloured ward on this map means that the ward has a relatively high value of `num_pois`.

It does **not** automatically mean that the ward has better accessibility, better services, or a higher quality of life.

## Raw counts can be misleading

Suppose:

```text
Ward A has 300 POIs
Ward B has 150 POIs
```

It is tempting to conclude:

> Ward A has a greater concentration of POIs.

But imagine:

```text
Ward A area = 30 km²
Ward B area = 5 km²
```

Then:

```text
Ward A: 300 / 30 = 10 POIs per km²
Ward B: 150 / 5 = 30 POIs per km²
```

Ward A has more POIs in total, but Ward B has a much higher **density**.

This shows why raw counts and density answer different questions.

## Calculate ward area

To calculate area correctly, the ward polygons must use a projected CRS whose units are metres.

Delhi lies in **UTM Zone 43N**, EPSG:32643.

We therefore create a projected copy:

In [ ]:
ward_utm = ward_map.to_crs(epsg=32643)

print(ward_utm.crs)

Now:

```python
ward_utm.geometry.area
```

returns area in square metres.

To convert square metres to square kilometres:

\[
1 	ext{ km}^2 = 1,000,000 	ext{ m}^2
\]

In [ ]:
ward_utm["area_km2"] = ward_utm.geometry.area / 1_000_000

ward_utm[["ward_name", "num_pois", "area_km2"]].head()

## Calculate POI density

We define:

\[
	ext{POI density}
=
rac{	ext{Number of POIs}}
{	ext{Ward area in km}^2}
\]

This gives the number of mapped POIs per square kilometre.

In [ ]:
ward_utm["poi_density"] = (
    ward_utm["num_pois"] /
    ward_utm["area_km2"]
)

ward_utm[
    ["ward_name", "num_pois", "area_km2", "poi_density"]
].head()

## Map the raw count

First, map the total number of POIs.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

ward_utm.plot(
    ax=ax,
    column="num_pois",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5
)

ax.set_title("Number of Mapped POIs by Ward")
ax.set_axis_off()

plt.show()

## Map POIs per km²

Now map the density.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

ward_utm.plot(
    ax=ax,
    column="poi_density",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5
)

ax.set_title("Mapped POIs per km² by Ward")
ax.set_axis_off()

plt.show()

## Compare the two maps

The two maps answer different questions.

### Count map

> Where are the largest numbers of mapped POIs located?

### Density map

> Where are mapped POIs most concentrated relative to ward area?

A ward can rank highly on one measure but not on the other.

This is why the choice of variable matters when creating a choropleth map.

## Important caution: density is not accessibility

Even POIs per km² is still a relatively simple measure.

It does not tell us:

- where people live within the ward;
- how far residents must actually travel;
- whether roads or paths connect people to the facilities;
- whether facilities have equal capacity or quality;
- whether the POI dataset is complete.

So we should describe the map carefully:

> **Mapped POI density by ward**

rather than making a stronger claim such as:

> **Accessibility by ward**

Later vector-processing and network-analysis methods can address some of these limitations.

## Class Exercise 14

Using only POIs where:

```python
category == "Green space"
```

perform the complete workflow:

1. Count the number of Green-space POIs in each ward.
2. Merge the counts back to the ward polygons.
3. Replace missing counts with `0`.
4. Reproject the ward layer to EPSG:32643.
5. Calculate ward area in km².
6. Calculate Green-space POIs per km².
7. Create:
   - one choropleth map of the raw Green-space POI count;
   - one choropleth map of Green-space POIs per km².
8. Briefly describe one difference you observe between the two maps.

The purpose is to connect everything from Lectures 15–17:

```text
spatial join
→ filter
→ group
→ merge
→ calculate
→ map
→ interpret
```

# Summary of Lectures 15–17

The main workflow introduced in these lectures is:

```text
POINT DATA
   +
POLYGON DATA
   ↓
SPATIAL JOIN
   ↓
Each point receives polygon attributes
   ↓
GROUP / COUNT
   ↓
Ward-level summary table
   ↓
NORMAL MERGE
   ↓
Values attached back to ward polygons
   ↓
CHOROPLETH MAP
   ↓
INTERPRETATION
```

The most important distinction is:

### Normal merge

```text
match using an attribute
```

Example:

```text
ward_id = ward_id
```

### Spatial join

```text
match using geometry
```

Example:

```text
POI point lies within ward polygon
```

The next set of lectures can move from spatial joins to **vector processing**, including operations such as buffer, clip, dissolve, overlay and proximity analysis.